# ASQ Explore

Load ASQ train data and inspect basic size.


In [1]:
from pathlib import Path
import json
import sys

print('Python executable:', sys.executable)
print('Using venv:', 'venv' in sys.executable)


Python executable: /Users/shayan/Projects/NarrativeSimilarity/venv/bin/python
Using venv: True


In [2]:
base = Path.cwd()
train_path = base / '..' / 'data' / 'ASQ' / 'asq_train.json'

if not train_path.exists():
    train_path = base / 'data' / 'ASQ' / 'asq_train.json'

print('Train file:', train_path)
with open(train_path, 'r', encoding='utf-8') as f:
    asq_train = json.load(f)

print('ASQ train length:', len(asq_train))


Train file: /Users/shayan/Projects/NarrativeSimilarity/src/../data/ASQ/asq_train.json
ASQ train length: 8865


In [3]:
# Basic schema exploration
print('Top-level object type:', type(asq_train).__name__)
print('Dataset length:', len(asq_train))

if len(asq_train) > 0 and isinstance(asq_train[0], dict):
    sample_keys = sorted(asq_train[0].keys())
    print('Keys in first record:')
    print(sample_keys)

    all_keys = set()
    for item in asq_train:
        if isinstance(item, dict):
            all_keys.update(item.keys())
    print('Union of keys across dataset:')
    print(sorted(all_keys))
else:
    print('Records are not dicts; skipping key extraction.')


Top-level object type: list
Dataset length: 8865
Keys in first record:
['id', 'label', 'narrative', 'qn1', 'qn2']
Union of keys across dataset:
['id', 'label', 'narrative', 'qn1', 'qn2']


In [4]:
# Show 2 sample data points
for i, item in enumerate(asq_train[:2]):
    print(f'\nSample #{i}')
    print(json.dumps(item, indent=2, ensure_ascii=False)[:4000])



Sample #0
{
  "narrative": "This might sound dumb but I've been in hospital 4 nights already and am meant to be going home tomorrow (I hope!). My roommate from the statt is lovely and I'm not sure if it was because I was so out of it or because she didn't snore but I haven't heard her snore before. Tonight she snores like a chainsaw and I CANNOT sleep at all. She's also developed a hacking cough that I think is related. It's my last night but if I don't sleep and am not well enough they might decide to keep me longer which I don't want. I also really need my sleep because I need to her better. The nurses are kinda bitchty and rude and I feel like they might say something loudly to or in front of my roommate if I complain. I can put up with a sleepless night normally but not when I'm this sick.",
  "qn1": "Is it rude to sneak out and ask a nurse if they have any advice?",
  "qn2": "How to deal with loud roommate?",
  "label": 0,
  "id": "9tfn8g"
}

Sample #1
{
  "narrative": "Hello /r/

In [5]:
# Show 2 sample data points
from textwrap import fill

def pretty_print_record(record, width=100):
    for k, v in record.items():
        if isinstance(v, str):
            print(f"{k}:")
            print(fill(v, width=width))
        else:
            print(f"{k}: {v}")
        print()

for i, item in enumerate(asq_train[:2]):
    print(f"\n{'='*20} Sample #{i} {'='*20}\n")
    pretty_print_record(item, width=100)



==================== Sample #0 ====================

narrative:
This might sound dumb but I've been in hospital 4 nights already and am meant to be going home
tomorrow (I hope!). My roommate from the statt is lovely and I'm not sure if it was because I was so
out of it or because she didn't snore but I haven't heard her snore before. Tonight she snores like
a chainsaw and I CANNOT sleep at all. She's also developed a hacking cough that I think is related.
It's my last night but if I don't sleep and am not well enough they might decide to keep me longer
which I don't want. I also really need my sleep because I need to her better. The nurses are kinda
bitchty and rude and I feel like they might say something loudly to or in front of my roommate if I
complain. I can put up with a sleepless night normally but not when I'm this sick.

qn1:
Is it rude to sneak out and ask a nurse if they have any advice?

qn2:
How to deal with loud roommate?

label: 0

id:
9tfn8g


==================== Samp

## Narrative Exploration

We want a quick statistical profile of the ASQ narratives before modeling.
The next cell computes basic length/distribution statistics such as sentence counts, word counts, and character counts (including percentiles).


In [6]:
import re
import pandas as pd

narr_df = pd.DataFrame(asq_train)
narr_df['narrative'] = narr_df['narrative'].fillna('').astype(str)

def count_sentences(text):
    parts = re.split(r'[.!?]+', text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)

def count_words(text):
    return len(re.findall(r"\b\w+\b", text))

narr_df['num_chars'] = narr_df['narrative'].str.len()
narr_df['num_words'] = narr_df['narrative'].apply(count_words)
narr_df['num_sentences'] = narr_df['narrative'].apply(count_sentences)

summary = narr_df[['num_chars', 'num_words', 'num_sentences']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T
summary = summary[['count', 'mean', 'std', 'min', '10%', '25%', '50%', '75%', '90%', 'max']]

print('Narrative basic statistics:')
display(summary)

print('Averages:')
print(f"  Avg sentences: {narr_df['num_sentences'].mean():.2f}")
print(f"  Avg words:     {narr_df['num_words'].mean():.2f}")
print(f"  Avg chars:     {narr_df['num_chars'].mean():.2f}")


Narrative basic statistics:


,count,mean,std,min,10%,25%,50%,75%,90%,max
num_chars,8865.0,722.874563,307.849148,192.0,329.0,462.0,696.0,955.0,1172.0,1549.0
num_words,8865.0,145.401918,61.042022,39.0,67.0,94.0,140.0,193.0,236.0,288.0
num_sentences,8865.0,8.014777,4.009987,1.0,3.0,5.0,7.0,10.0,14.0,36.0


Averages:
  Avg sentences: 8.01
  Avg words:     145.40
  Avg chars:     722.87


## Topic Exploration

To get a qualitative sense of what the stories are about, we can:
- inspect frequent unigrams/bigrams after basic stopword filtering, and
- fit a simple unsupervised topic model (NMF on TF-IDF) and print top words per topic.


In [7]:
# Frequent terms + simple topic model for narrative themes
from collections import Counter
import re

texts = narr_df['narrative'].fillna('').astype(str).tolist()

# Lightweight tokenizer and stopword list (kept local so this cell works without extra deps).
stopwords = {
    'the','a','an','and','or','but','if','then','so','because','as','to','of','in','on','at','for','from','with','without',
    'is','am','are','was','were','be','been','being','do','does','did','doing','have','has','had','having',
    'i','me','my','mine','myself','we','our','ours','ourselves','you','your','yours','yourself','yourselves',
    'he','him','his','she','her','hers','they','them','their','theirs','it','its','this','that','these','those',
    'not','no','yes','just','really','very','also','can','could','would','should','will','im','ive','id','dont','didnt','cant',
    'about','into','over','under','again','still','more','most','some','any','all','only','than','too','when','while','where',
    'what','which','who','whom','why','how'
}

def tokenize(text):
    toks = re.findall(r"[a-zA-Z']+", text.lower())
    return [t for t in toks if len(t) > 2 and t not in stopwords]

unigram_counter = Counter()
bigram_counter = Counter()
for t in texts:
    toks = tokenize(t)
    unigram_counter.update(toks)
    bigram_counter.update([' '.join(pair) for pair in zip(toks, toks[1:])])

print('Top 25 unigrams:')
for w, c in unigram_counter.most_common(25):
    print(f'  {w:20s} {c}')

print('\nTop 25 bigrams:')
for w, c in bigram_counter.most_common(25):
    print(f'  {w:25s} {c}')

# Optional: NMF topic modeling if sklearn is available.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import NMF

    n_topics = 8
    tfidf = TfidfVectorizer(stop_words='english', min_df=10, max_df=0.6, ngram_range=(1,2))
    X = tfidf.fit_transform(texts)

    nmf = NMF(n_components=n_topics, random_state=42, init='nndsvda', max_iter=400)
    W = nmf.fit_transform(X)
    H = nmf.components_
    terms = tfidf.get_feature_names_out()

    print('\nNMF topics (top 10 terms each):')
    for k, topic_vec in enumerate(H):
        top_idx = topic_vec.argsort()[-10:][::-1]
        top_terms = [terms[i] for i in top_idx]
        print(f'  Topic {k+1}: ' + ', '.join(top_terms))

    dominant_topic = W.argmax(axis=1)
    topic_counts = Counter(dominant_topic.tolist())
    print('\nDominant-topic counts:')
    for k in range(n_topics):
        print(f'  Topic {k+1}: {topic_counts.get(k, 0)}')

except Exception as e:
    print('\nNMF topic modeling skipped:', e)
    print('Keyword and bigram frequency output above still gives a useful theme overview.')


Top 25 unigrams:
  i'm                  11539
  like                 6788
  don't                6515
  out                  5517
  want                 5418
  know                 5216
  get                  4687
  i've                 4389
  time                 4244
  now                  3822
  it's                 3720
  feel                 3544
  one                  3119
  work                 3005
  job                  2970
  people               2837
  friends              2731
  year                 2624
  going                2534
  other                2533
  there                2513
  school               2478
  even                 2455
  years                2343
  think                2295

Top 25 bigrams:
  don't know                1693
  feel like                 1485
  don't want                1408
  i'm sure                  641
  high school               637
  right now                 582
  year old                  470
  each other                452
  i'm 